# Tugas SQL: Analisis Penjualan Showroom Mobil

Notebook ini mengerjakan 10 latihan SQL dan 3 pertanyaan analisis penutup menggunakan dataset **`Day 2 - showroom_mobil_10cabang_clean.csv`**.

**Catatan:** Tiga analisis penutup hanya menggunakan transaksi dengan `status = 'completed'`, sesuai instruksi tugas.

## 0. Persiapan data

Jalankan sel berikut. Jika file belum tersedia di Google Colab, kotak unggah akan muncul secara otomatis.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

file_names = [
    "Day 2 - showroom_mobil_10cabang_clean.csv",
    "showroom_mobil_10cabang_10k (1).xlsx",
]

candidates = []
for name in file_names:
    candidates.extend([Path(name), Path("/content") / name])

data_path = next((path for path in candidates if path.exists()), None)

if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        data_path = Path(uploaded_name)
    except ImportError as exc:
        raise FileNotFoundError(
            "Letakkan file CSV/XLSX pada folder yang sama dengan notebook."
        ) from exc

if data_path.suffix.lower() == ".csv":
    df = pd.read_csv(data_path)
else:
    df = pd.read_excel(data_path)

required_columns = {
    "sales_date", "order_id", "customer_name", "branch", "product_name",
    "category", "color", "price", "quantity", "payment_type", "trade_in",
    "discount", "total", "total_sales", "status", "branch_address"
}
missing = required_columns.difference(df.columns)
assert not missing, f"Kolom yang belum tersedia: {sorted(missing)}"

conn = sqlite3.connect(":memory:")
df.to_sql("showroom_mobil", conn, index=False, if_exists="replace")

def run_query(sql):
    return pd.read_sql_query(sql, conn)

print(f"File digunakan : {data_path.name}")
print(f"Jumlah baris   : {len(df):,}")
print(f"Jumlah kolom   : {len(df.columns)}")
df.head()

File digunakan : Day 2 - showroom_mobil_10cabang_clean.csv
Jumlah baris   : 10,000
Jumlah kolom   : 16


,sales_date,order_id,customer_name,branch,product_name,category,color,price,quantity,payment_type,trade_in,discount,total,total_sales,status,branch_address
0,12/13/2025,TRX00001,Customer_913,Jakarta,Daihatsu Xenia 1.5 R CVT,MPV,Silver Metalik,240000000,1,Cash + Trade In,74000000,0,240000000,166000000,completed,"Jl. MT. Haryono No. 12, Jakarta"
1,10/15/2025,TRX00002,Customer_261,Jakarta,Toyota Rush 1.5 S GR Sport CVT,SUV,Hitam Metalik,300000000,1,Kredit,0,0,300000000,300000000,completed,"Jl. MT. Haryono No. 12, Jakarta"
2,1/4/2025,TRX00003,Customer_4465,Kendari,Honda BR-V 1.5 E CVT,MPV,Merah,310000000,1,Kredit + Trade In,111000000,0,310000000,199000000,completed,"Jl. Ahmad Yani No. 33, Kendari"
3,2/17/2025,TRX00004,Customer_2788,Padang,Honda Brio Satya E CVT,Hatchback,Merah,190000000,1,Kredit,0,0,190000000,190000000,completed,"Jl. Khatib Sulaiman No. 10, Padang"
4,10/2/2025,TRX00005,Customer_2941,Karawang,Isuzu D-Max Double Cabin 4x4 AT,Pickup,Hitam Metalik,520000000,1,Cash + Trade In,93000000,0,520000000,427000000,refund,"Jl. Ahmad Yani No. 5, Karawang"


## Latihan SQL 1–10

### 1. Filter Nama Pelanggan Unik

Menampilkan seluruh `customer_name` yang unik. Hasil dataset ini berisi **3.977 pelanggan unik**.

In [ ]:
query_01 = '''
SELECT DISTINCT customer_name
FROM showroom_mobil
ORDER BY customer_name;
'''
result_01 = run_query(query_01)
print(f"Jumlah pelanggan unik: {len(result_01):,}")
result_01

Jumlah pelanggan unik: 3,977


,customer_name
0,Customer_1
1,Customer_10
2,Customer_100
3,Customer_1000
4,Customer_1001
...,...
3972,Customer_995
3973,Customer_996
3974,Customer_997
3975,Customer_998


### 2. Rekap Transaksi Kredit

Menampilkan `order_id`, `branch`, dan `total_sales` untuk pembayaran **Kredit**.

In [ ]:
query_02 = '''
SELECT order_id, branch, total_sales
FROM showroom_mobil
WHERE payment_type = 'Kredit'
ORDER BY order_id;
'''
result_02 = run_query(query_02)
print(f"Jumlah transaksi Kredit: {len(result_02):,}")
result_02

Jumlah transaksi Kredit: 4,253


,order_id,branch,total_sales
0,TRX00002,Jakarta,300000000
1,TRX00004,Padang,190000000
2,TRX00006,Bandung,250000000
3,TRX00008,Pontianak,580000000
4,TRX00009,Semarang,310000000
...,...,...,...
4248,TRX09987,Pontianak,240000000
4249,TRX09988,Padang,1040000000
4250,TRX09989,Bandung,580000000
4251,TRX09996,Karawang,240000000


### 3. Pembelian dengan Quantity Terbanyak

Menampilkan lima transaksi dengan `quantity` terbesar. `order_id` digunakan sebagai pengurutan tambahan agar hasil konsisten ketika ada nilai quantity yang sama.

In [ ]:
query_03 = '''
SELECT order_id, customer_name, branch, product_name, quantity
FROM showroom_mobil
ORDER BY quantity DESC, order_id ASC
LIMIT 5;
'''
run_query(query_03)

,order_id,customer_name,branch,product_name,quantity
0,TRX00069,Customer_671,Malang,Toyota Avanza 1.5 G CVT,3
1,TRX00089,Customer_1223,Batam,Daihatsu Terios 1.5 R CVT,3
2,TRX00636,Customer_2883,Karawang,Daihatsu Terios 1.5 R CVT,3
3,TRX01301,Customer_1271,Bandung,Daihatsu Xenia 1.5 R CVT,3
4,TRX01374,Customer_634,Palembang,Nissan Livina VE AT,3


### 4. Model Mobil Berwarna Hitam Metalik

Menampilkan `product_name` unik dengan warna **Hitam Metalik**.

In [ ]:
query_04 = '''
SELECT DISTINCT product_name
FROM showroom_mobil
WHERE color = 'Hitam Metalik'
ORDER BY product_name;
'''
run_query(query_04)

,product_name
0,Daihatsu Terios 1.5 R CVT
1,Daihatsu Xenia 1.5 R CVT
2,Honda BR-V 1.5 E CVT
3,Honda CR-V 1.5 Turbo RS
4,Honda HR-V 1.5 E CVT
5,Isuzu D-Max Double Cabin 4x4 AT
6,Mitsubishi Pajero Sport Dakar 4x2 AT
7,Mitsubishi Xpander Ultimate CVT
8,Nissan Livina VE AT
9,Suzuki Ertiga GX AT


### 5. Komposisi Metode Pembayaran

Menghitung jumlah transaksi pada setiap `payment_type`, lalu mengurutkannya dari yang terbanyak.

In [ ]:
query_05 = '''
SELECT payment_type,
       COUNT(*) AS jumlah_transaksi
FROM showroom_mobil
GROUP BY payment_type
ORDER BY jumlah_transaksi DESC;
'''
run_query(query_05)

,payment_type,jumlah_transaksi
0,Kredit,4253
1,Cash,2224
2,Kredit + Trade In,1999
3,Cash + Trade In,1524


### 6. Evaluasi Rata-rata Diskon

Menghitung rata-rata `discount` untuk setiap `category` dan mengurutkannya dari yang terbesar.

In [ ]:
query_06 = '''
SELECT category,
       ROUND(AVG(discount), 2) AS rata_rata_discount
FROM showroom_mobil
GROUP BY category
ORDER BY rata_rata_discount DESC;
'''
run_query(query_06)

,category,rata_rata_discount
0,Hatchback,"55,876.69"
1,City Car (EV),"45,360.82"
2,Pickup,"38,759.69"
3,MPV,"29,629.63"
4,SUV,"28,696.60"


### 7. Cabang dengan Pendapatan di Atas Rp50 Miliar

Karena nilai yang disaring merupakan hasil agregasi `SUM(total_sales)`, kondisi diletakkan pada klausa `HAVING`, bukan `WHERE`.

In [ ]:
query_07 = '''
SELECT branch,
       SUM(total_sales) AS total_pendapatan
FROM showroom_mobil
GROUP BY branch
HAVING SUM(total_sales) > 50000000000
ORDER BY total_pendapatan DESC;
'''
run_query(query_07)

,branch,total_pendapatan
0,Malang,325311000000
1,Jakarta,322346000000
2,Padang,321583000000
3,Palembang,314018000000
4,Bandung,309919000000
5,Batam,307843000000
6,Kendari,304402000000
7,Semarang,300335000000
8,Pontianak,298650000000
9,Karawang,296360000000


### 8. Investigasi Diskon di Atas Rata-rata

Subquery menghitung rata-rata diskon seluruh transaksi. Query utama mengambil transaksi dengan diskon yang lebih besar dari nilai tersebut.

In [ ]:
query_08 = '''
SELECT order_id, product_name, discount
FROM showroom_mobil
WHERE discount > (
    SELECT AVG(discount)
    FROM showroom_mobil
)
ORDER BY discount DESC, order_id ASC;
'''
result_08 = run_query(query_08)
print(f"Rata-rata discount seluruh transaksi: Rp{df['discount'].mean():,.2f}")
print(f"Jumlah transaksi di atas rata-rata  : {len(result_08):,}")
result_08

Rata-rata discount seluruh transaksi: Rp31,800.00
Jumlah transaksi di atas rata-rata  : 318


,order_id,product_name,discount
0,TRX00021,Suzuki Ertiga GX AT,1000000
1,TRX00032,Toyota Rush 1.5 S GR Sport CVT,1000000
2,TRX00080,Wuling Air ev Long Range,1000000
3,TRX00083,Daihatsu Xenia 1.5 R CVT,1000000
4,TRX00086,Wuling Air ev Long Range,1000000
...,...,...,...
313,TRX09808,Isuzu D-Max Double Cabin 4x4 AT,1000000
314,TRX09811,Honda HR-V 1.5 E CVT,1000000
315,TRX09823,Honda Brio Satya E CVT,1000000
316,TRX09856,Toyota Innova Zenix 2.0 V CVT,1000000


### 9. Rata-rata Pendapatan per Kategori untuk Transaksi Completed

Subquery menyaring `status = 'completed'` terlebih dahulu, kemudian query utama menghitung rata-rata `total_sales` per kategori.

In [ ]:
query_09 = '''
SELECT category,
       ROUND(AVG(total_sales), 2) AS rata_rata_total_sales
FROM (
    SELECT category, total_sales
    FROM showroom_mobil
    WHERE status = 'completed'
) AS transaksi_selesai
GROUP BY category
ORDER BY rata_rata_total_sales DESC;
'''
run_query(query_09)

,category,rata_rata_total_sales
0,Pickup,"472,825,221.24"
1,SUV,"367,695,154.61"
2,MPV,"255,047,142.86"
3,City Car (EV),"202,664,383.56"
4,Hatchback,"148,799,573.56"


### 10. Tiga Model Mobil Terlaris dengan CTE

CTE menghitung jumlah transaksi per `product_name`, lalu query utama mengambil tiga model dengan jumlah transaksi terbanyak.

In [ ]:
query_10 = '''
WITH transaksi_per_model AS (
    SELECT product_name,
           COUNT(*) AS jumlah_transaksi
    FROM showroom_mobil
    GROUP BY product_name
)
SELECT product_name, jumlah_transaksi
FROM transaksi_per_model
ORDER BY jumlah_transaksi DESC, product_name ASC
LIMIT 3;
'''
run_query(query_10)

,product_name,jumlah_transaksi
0,Honda HR-V 1.5 E CVT,557
1,Toyota Fortuner 2.4 VRZ 4x2 AT,528
2,Hyundai Creta Prime AT,526


# Analisis Penutup

Semua query pada bagian ini memakai `status = 'completed'`.

## 1. Customer yang Paling Sering Menggunakan Trade-In

Hasilnya **seri**: `Customer_3155` dan `Customer_254` sama-sama memiliki **5 transaksi completed dengan Trade-In**. Jika diperlukan satu prioritas berdasarkan nilai `total_sales`, `Customer_3155` berada di atas dengan Rp1.082.000.000, dibandingkan Rp803.000.000 milik `Customer_254`.

In [ ]:
query_analisis_01 = '''
WITH trade_in_customer AS (
    SELECT customer_name,
           COUNT(*) AS jumlah_transaksi_trade_in,
           SUM(quantity) AS jumlah_unit,
           SUM(total_sales) AS total_sales
    FROM showroom_mobil
    WHERE status = 'completed'
      AND payment_type LIKE '%Trade In%'
    GROUP BY customer_name
)
SELECT customer_name,
       jumlah_transaksi_trade_in,
       jumlah_unit,
       total_sales
FROM trade_in_customer
WHERE jumlah_transaksi_trade_in = (
    SELECT MAX(jumlah_transaksi_trade_in)
    FROM trade_in_customer
)
ORDER BY total_sales DESC;
'''
run_query(query_analisis_01)

,customer_name,jumlah_transaksi_trade_in,jumlah_unit,total_sales
0,Customer_3155,5,6,1082000000
1,Customer_254,5,5,803000000


## 2. Cabang yang Paling Bergantung pada Trade-In

Cabang **Karawang** memiliki persentase transaksi Trade-In tertinggi, yaitu **37,76%** atau **330 dari 874 transaksi completed**. Persentase dihitung terhadap seluruh transaksi completed di masing-masing cabang, bukan terhadap jumlah Trade-In nasional.

In [ ]:
query_analisis_02 = '''
WITH trade_in_per_cabang AS (
    SELECT branch,
           COUNT(*) AS transaksi_completed,
           SUM(
               CASE
                   WHEN payment_type LIKE '%Trade In%' THEN 1
                   ELSE 0
               END
           ) AS transaksi_trade_in
    FROM showroom_mobil
    WHERE status = 'completed'
    GROUP BY branch
)
SELECT branch,
       transaksi_trade_in,
       transaksi_completed,
       ROUND(
           100.0 * transaksi_trade_in / transaksi_completed,
           2
       ) AS persentase_trade_in
FROM trade_in_per_cabang
ORDER BY persentase_trade_in DESC, transaksi_trade_in DESC;
'''
run_query(query_analisis_02)

,branch,transaksi_trade_in,transaksi_completed,persentase_trade_in
0,Karawang,330,874,37.76
1,Jakarta,351,938,37.42
2,Kendari,323,882,36.62
3,Pontianak,322,880,36.59
4,Semarang,308,865,35.61
5,Bandung,313,892,35.09
6,Batam,306,893,34.27
7,Palembang,297,874,33.98
8,Padang,313,942,33.23
9,Malang,302,926,32.61


## 3. Rekomendasi Penjualan untuk Cabang Bandung

Analisis berikut melihat kategori, model, warna, dan kombinasi model–warna pada transaksi completed di Bandung.

In [ ]:
query_bandung_kategori = '''
SELECT category,
       COUNT(*) AS jumlah_transaksi,
       SUM(quantity) AS jumlah_unit,
       SUM(total_sales) AS total_sales
FROM showroom_mobil
WHERE status = 'completed'
  AND branch = 'Bandung'
GROUP BY category
ORDER BY jumlah_unit DESC, total_sales DESC;
'''
run_query(query_bandung_kategori)

,category,jumlah_transaksi,jumlah_unit,total_sales
0,SUV,403,416,149539000000
1,MPV,344,362,89141000000
2,City Car (EV),52,56,11737000000
3,Pickup,49,50,21927000000
4,Hatchback,44,47,6769000000


In [ ]:
query_bandung_model = '''
SELECT product_name,
       category,
       COUNT(*) AS jumlah_transaksi,
       SUM(quantity) AS jumlah_unit,
       SUM(total_sales) AS total_sales
FROM showroom_mobil
WHERE status = 'completed'
  AND branch = 'Bandung'
GROUP BY product_name, category
ORDER BY total_sales DESC, jumlah_unit DESC;
'''
run_query(query_bandung_model).head(10)

,product_name,category,jumlah_transaksi,jumlah_unit,total_sales
0,Honda CR-V 1.5 Turbo RS,SUV,46,48,28373000000
1,Mitsubishi Pajero Sport Dakar 4x2 AT,SUV,47,48,25437000000
2,Toyota Fortuner 2.4 VRZ 4x2 AT,SUV,45,47,25405000000
3,Isuzu D-Max Double Cabin 4x4 AT,Pickup,49,50,21927000000
4,Toyota Innova Zenix 2.0 V CVT,MPV,46,48,19012000000
5,Honda HR-V 1.5 E CVT,SUV,44,46,16223000000
6,Mitsubishi Xpander Ultimate CVT,MPV,51,53,14339000000
7,Daihatsu Terios 1.5 R CVT,SUV,51,53,12840000000
8,Wuling Air ev Long Range,City Car (EV),52,56,11737000000
9,Wuling Almaz RS Pro AT,SUV,40,41,10924000000


In [ ]:
query_bandung_warna = '''
SELECT color,
       COUNT(*) AS jumlah_transaksi,
       SUM(quantity) AS jumlah_unit,
       SUM(total_sales) AS total_sales
FROM showroom_mobil
WHERE status = 'completed'
  AND branch = 'Bandung'
GROUP BY color
ORDER BY jumlah_unit DESC, total_sales DESC;
'''
run_query(query_bandung_warna).head(10)

,color,jumlah_transaksi,jumlah_unit,total_sales
0,Hitam Metalik,196,203,65824000000
1,Putih Mutiara,150,159,50142000000
2,Silver Metalik,103,104,26364000000
3,Merah,83,86,22229000000
4,Putih,52,55,13321000000
5,Coklat Metalik,49,54,15118000000
6,Abu-abu Metalik,45,46,20220000000
7,Hitam Phantom,30,30,7500000000
8,Putih Pearl,24,26,10289000000
9,Putih Atlas,24,25,5929000000


In [ ]:
query_bandung_kombinasi = '''
SELECT product_name,
       color,
       COUNT(*) AS jumlah_transaksi,
       SUM(quantity) AS jumlah_unit,
       SUM(total_sales) AS total_sales
FROM showroom_mobil
WHERE status = 'completed'
  AND branch = 'Bandung'
GROUP BY product_name, color
ORDER BY jumlah_unit DESC, total_sales DESC
LIMIT 10;
'''
run_query(query_bandung_kombinasi)

,product_name,color,jumlah_transaksi,jumlah_unit,total_sales
0,Honda CR-V 1.5 Turbo RS,Hitam Metalik,19,20,11590000000
1,Hyundai Stargazer Prime AT,Hitam Phantom,20,20,4859000000
2,Mitsubishi Pajero Sport Dakar 4x2 AT,Hitam Metalik,18,19,9736000000
3,Wuling Air ev Long Range,Putih,18,19,3586000000
4,Honda CR-V 1.5 Turbo RS,Putih Mutiara,17,18,10737000000
5,Toyota Innova Zenix 2.0 V CVT,Putih Mutiara,16,18,7149000000
6,Daihatsu Terios 1.5 R CVT,Merah,18,18,4956000000
7,Mitsubishi Pajero Sport Dakar 4x2 AT,Abu-abu Quartz,17,17,9153000000
8,Nissan Livina VE AT,Hitam Metalik,16,17,3729000000
9,Daihatsu Xenia 1.5 R CVT,Coklat Metalik,13,16,3379000000


### Kesimpulan Rekomendasi Bandung

Saya merekomendasikan cabang Bandung memprioritaskan **Honda CR-V 1.5 Turbo RS**, terutama warna **Hitam Metalik**.

Alasannya:

- Kategori **SUV** menjadi kategori terkuat di Bandung dengan **416 unit** dan `total_sales` **Rp149,539 miliar**.
- Honda CR-V menghasilkan `total_sales` model tertinggi, yaitu **Rp28,373 miliar**, dari **48 unit**.
- Kombinasi **Honda CR-V Hitam Metalik** menjadi kombinasi model–warna terlaris dengan **20 unit** dan `total_sales` **Rp11,590 miliar**.
- Warna **Hitam Metalik** juga menjadi warna terlaris secara keseluruhan di Bandung dengan **203 unit**.

Jika tujuan cabang lebih menekankan volume unit daripada nilai penjualan, **Wuling Air ev Long Range** dapat menjadi pilihan kedua karena mencatat volume tertinggi, yaitu **56 unit**. Namun, untuk gabungan kekuatan kategori, nilai penjualan, dan preferensi warna, Honda CR-V Hitam Metalik merupakan rekomendasi yang paling kuat.

> Catatan: Dataset tidak memiliki kolom harga pokok atau laba. Karena itu, rekomendasi ini menggunakan jumlah unit dan `total_sales`, bukan margin keuntungan.

## Ringkasan Jawaban Akhir

1. **Customer paling sering memakai Trade-In:** seri antara `Customer_3155` dan `Customer_254`, masing-masing 5 transaksi completed.
2. **Cabang paling bergantung pada Trade-In:** **Karawang**, 37,76% (330 dari 874 transaksi completed).
3. **Rekomendasi Bandung:** **Honda CR-V 1.5 Turbo RS warna Hitam Metalik**, karena didukung kategori SUV yang dominan, `total_sales` model tertinggi, dan kombinasi model–warna terlaris.